# Recompra

Como a recompra observada varia conforme a avaliação da primeira compra entregue? Considera pelo menos 90 dias observáveis e novos pedidos após a compra de referência.

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)


In [ ]:
from pathlib import Path

# Procura a raiz a partir da pasta atual; funciona também nas cópias em outputs/notebooks.
RAIZ_PROJETO = next(
    (pasta for pasta in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (pasta / 'executar_pipeline.py').is_file() and (pasta / 'src').is_dir()),
    None
)
if RAIZ_PROJETO is None:
    raise FileNotFoundError('Execute o notebook dentro da pasta do projeto ou de uma subpasta.')

CAMINHO_PROCESSADOS = RAIZ_PROJETO / 'data' / 'processed'

df_base = pd.read_parquet(f'{CAMINHO_PROCESSADOS}/base_analitica_pedidos.parquet')
df_itens_pedidos = pd.read_parquet(f'{CAMINHO_PROCESSADOS}/itens_pedidos.parquet')


In [ ]:
colunas_base = [
    'order_id',
    'customer_id',
    'customer_unique_id',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'review_id',
    'review_score',
    'review_event_timestamp',
    'review_ruim',
    'review_neutra',
    'review_boa',
    'possui_avaliacao',
    'possui_itens',
    'pedido_entregue',
    'is_late',
    'delivery_delay_days'
]


df_recompra = df_base[colunas_base].copy()
display(df_recompra.isna().sum())


In [ ]:
DATA_FIM_OBSERVACAO = df_recompra['order_purchase_timestamp'].max()
JANELA_RECOMPRA = pd.Timedelta(days=90)

colunas_primeira_compra = ["customer_unique_id","order_id","order_purchase_timestamp","order_delivered_customer_date","review_id","review_score","review_event_timestamp","review_ruim","review_neutra","review_boa","possui_avaliacao","pedido_entregue","is_late","delivery_delay_days"]

df_primeira_compra = (
    df_recompra.loc[
        lambda dados: dados['pedido_entregue'],
        colunas_primeira_compra
    ]
    .sort_values(['order_purchase_timestamp', 'order_id'])
    .drop_duplicates(subset='customer_unique_id', keep='first')
    .dropna(subset=['order_delivered_customer_date'])
    .rename(columns={
    "order_id": "first_order_id",
    "order_purchase_timestamp": "first_order_purchase_timestamp",
    "order_delivered_customer_date": "first_order_delivered_customer_date",
    "review_id": "first_review_id",
    "review_score": "first_order_review_score",
    "review_event_timestamp": "first_order_review_event_timestamp",
    "review_ruim": "first_review_ruim",
    "review_neutra": "first_review_neutra",
    "review_boa": "first_review_boa",
    "possui_avaliacao": "first_possui_avaliacao",
    "pedido_entregue": "first_pedido_entregue",
    "is_late": "first_is_late",
    "delivery_delay_days": "first_delivery_delay_days"
})
    .reset_index(drop=True)
)

df_primeira_compra['fim_janela_recompra'] = (
    df_primeira_compra['first_order_delivered_customer_date']
    + JANELA_RECOMPRA
)

df_primeira_compra_elegivel = (
    df_primeira_compra.loc[
        lambda dados: dados['fim_janela_recompra'].le(DATA_FIM_OBSERVACAO)
    ]
    .copy()
)

assert df_primeira_compra_elegivel['customer_unique_id'].is_unique

display(df_primeira_compra.isna().sum())
display(len(df_primeira_compra))
display(len(df_primeira_compra_elegivel))


In [ ]:
assert df_primeira_compra_elegivel['customer_unique_id'].is_unique
assert df_base['order_id'].is_unique

df_recompra = (
    df_base[colunas_primeira_compra]
    .merge(
        df_primeira_compra_elegivel[
            ['customer_unique_id', 'first_order_id', 'first_order_purchase_timestamp']
        ],
        on='customer_unique_id',
        how='inner',
        validate='many_to_one'
    )
    # Exclui pedidos anteriores ou simultâneos à compra entregue de referência.
    .loc[
        lambda dados: (
            dados['order_id'].ne(dados['first_order_id'])
            & dados['order_purchase_timestamp'].gt(
                dados['first_order_purchase_timestamp']
            )
        )
    ]
    # A recompra não exige entrega nem avaliação e não tem prazo máximo.
    .sort_values(['customer_unique_id', 'order_purchase_timestamp', 'order_id'])
    .drop_duplicates(subset='customer_unique_id', keep='first')
    .rename(columns={
    "order_id": "next_order_id",
    "order_purchase_timestamp": "next_order_purchase_timestamp",
    "order_delivered_customer_date": "next_order_delivered_customer_date",
    "review_id": "next_review_id",
    "review_score": "next_order_review_score",
    "review_event_timestamp": "next_order_review_event_timestamp",
    "review_ruim": "next_review_ruim",
    "review_neutra": "next_review_neutra",
    "review_boa": "next_review_boa",
    "possui_avaliacao": "next_possui_avaliacao",
    "pedido_entregue": "next_pedido_entregue",
    "is_late": "next_is_late",
    "delivery_delay_days": "next_delivery_delay_days"
})
    .reset_index(drop=True)
    .copy()
)

assert df_recompra['customer_unique_id'].is_unique
assert df_recompra['next_order_id'].ne(df_recompra['first_order_id']).all()
assert (
    df_recompra['next_order_purchase_timestamp']
    .gt(df_recompra['first_order_purchase_timestamp']).all()
)

# As colunas first usadas no filtro continuam na tabela da primeira compra.
df_recompra = df_recompra.drop(
    columns=['first_order_id', 'first_order_purchase_timestamp']
)

display(df_recompra.head())
print('Clientes com recompra:', len(df_recompra))


In [ ]:
# O left merge preserva os clientes que não recompraram no denominador.
df_analise_recompra = (
    df_primeira_compra_elegivel
    .merge(
        df_recompra,
        on='customer_unique_id',
        how='left',
        validate='one_to_one'
    )
)

df_analise_recompra['recomprou'] = (
    df_analise_recompra['next_order_id'].notna()
)
df_analise_recompra['dias_ate_recompra'] = (
    df_analise_recompra['next_order_purchase_timestamp']
    - df_analise_recompra['first_order_purchase_timestamp']
).dt.total_seconds().div(86400)

assert df_analise_recompra['customer_unique_id'].is_unique
assert len(df_analise_recompra) == len(df_primeira_compra_elegivel)
assert df_analise_recompra.loc[
    df_analise_recompra['recomprou'], 'dias_ate_recompra'
].gt(0).all()


In [ ]:
def analisar_taxa_recompra(dados, grupo='Geral'):
    """Retorna os clientes do grupo e uma linha com a taxa de recompra."""
    nota = dados['first_order_review_score']

    filtros = {
        'Geral': pd.Series(True, index=dados.index),
        'Ruim': nota.isin([1, 2]),
        'Neutra': nota.eq(3).fillna(False),
        'Boa': nota.isin([4, 5]),
        'Qualquer avaliação': nota.isin([1, 2, 3, 4, 5]),
        'Sem avaliação': nota.isna()
    }
    if grupo not in filtros:
        raise ValueError(f'Grupo inválido: {grupo}.')

    resultado = dados.loc[filtros[grupo]].copy()
    total_clientes = len(resultado)
    clientes_que_recompraram = int(resultado['recomprou'].sum())

    resumo = pd.DataFrame([{
        'grupo': grupo,
        'total_clientes': total_clientes,
        'clientes_que_recompraram': clientes_que_recompraram,
        'clientes_que_nao_recompraram': total_clientes - clientes_que_recompraram,
        # Sem clientes, a taxa é desconhecida, não zero.
        'taxa_recompra': (
            clientes_que_recompraram / total_clientes
            if total_clientes else float('nan')
        )
    }])
    return resultado, resumo


In [ ]:
df_resultado_recompra_geral, resumo_geral = analisar_taxa_recompra(
    df_analise_recompra, 'Geral'
)
df_resultado_recompra_ruim, resumo_ruim = analisar_taxa_recompra(
    df_analise_recompra, 'Ruim'
)
df_resultado_recompra_neutra, resumo_neutra = analisar_taxa_recompra(
    df_analise_recompra, 'Neutra'
)
df_resultado_recompra_boa, resumo_boa = analisar_taxa_recompra(
    df_analise_recompra, 'Boa'
)
df_resultado_recompra_geral_avaliacoes, resumo_avaliados = analisar_taxa_recompra(
    df_analise_recompra, 'Qualquer avaliação'
)
df_resultado_recompra_sem_avaliacao, resumo_sem_avaliacao = analisar_taxa_recompra(
    df_analise_recompra, 'Sem avaliação'
)

df_comparacao_taxas_recompra = pd.concat(
    [
        resumo_geral, resumo_ruim, resumo_neutra, resumo_boa,
        resumo_avaliados, resumo_sem_avaliacao
    ],
    ignore_index=True
)

display(
    df_comparacao_taxas_recompra.style.format(
        {'taxa_recompra': '{:.2%}'}, na_rep='Sem clientes'
    )
)


In [ ]:
# Tempo contado desde a compra de referência, somente entre quem recomprou.
resultados_por_grupo = {
    'Geral': df_resultado_recompra_geral,
    'Ruim': df_resultado_recompra_ruim,
    'Neutra': df_resultado_recompra_neutra,
    'Boa': df_resultado_recompra_boa,
    'Qualquer avaliação': df_resultado_recompra_geral_avaliacoes,
    'Sem avaliação': df_resultado_recompra_sem_avaliacao
}

linhas_tempo_recompra = []
for grupo, resultado in resultados_por_grupo.items():
    dias = resultado.loc[resultado['recomprou'], 'dias_ate_recompra']
    linhas_tempo_recompra.append({
        'grupo': grupo,
        'quantidade': dias.count(),
        'media': dias.mean(),
        'mediana': dias.median(),
        'minimo': dias.min(),
        'maximo': dias.max(),
        'percentil_25': dias.quantile(0.25),
        'percentil_75': dias.quantile(0.75)
    })

df_resumo_tempo_recompra = pd.DataFrame(linhas_tempo_recompra)
display(
    df_resumo_tempo_recompra.style.format({
        'media': '{:.2f}', 'mediana': '{:.2f}',
        'minimo': '{:.2f}', 'maximo': '{:.2f}',
        'percentil_25': '{:.2f}', 'percentil_75': '{:.2f}'
    }, na_rep='Sem recompras')
)


In [ ]:
# Os grupos de nota são exclusivos e, junto aos sem avaliação,
# devem reproduzir os totais da população geral.
assert df_analise_recompra['first_order_review_score'].dropna().isin(
    [1, 2, 3, 4, 5]
).all()

resumos_exclusivos = pd.concat(
    [resumo_ruim, resumo_neutra, resumo_boa, resumo_sem_avaliacao]
)
assert resumos_exclusivos['total_clientes'].sum() == len(df_analise_recompra)
assert resumos_exclusivos['clientes_que_recompraram'].sum() == (
    df_analise_recompra['recomprou'].sum()
)
assert df_comparacao_taxas_recompra['taxa_recompra'].dropna().between(0, 1).all()
print('Validações das taxas concluídas.')


## Gráficos

Montagem dos gráficos da análise.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from io import BytesIO

plt.rcParams['text.parse_math'] = False
PASTA_GRAFICOS = RAIZ_PROJETO / 'outputs' / 'graficos'
graficos_exportar = {}


def registrar_grafico(figura, caminho, dpi=180, bbox_inches='tight'):
    # Captura antes de fechar a figura; grava os arquivos apenas na seção final.
    buffer = BytesIO()
    figura.savefig(buffer, format='png', dpi=dpi, bbox_inches=bbox_inches)
    nome = Path(caminho).name
    if nome in graficos_exportar:
        raise ValueError(f'Nome de gráfico repetido: {nome}')
    graficos_exportar[nome] = buffer.getvalue()


def concluir_grafico(figura, nome, nota):
    figura.text(0.01, 0.01, nota, fontsize=9, color='#555555')
    figura.tight_layout(rect=[0, 0.09, 1, 0.97])
    registrar_grafico(figura, nome)
    plt.show()
    plt.close(figura)


def grafico_situacoes(tabela, grupo, metrica, denominador, titulo, nome, nota):
    # Ausência de grupo permanece NaN: não representa uma taxa observada de zero.
    grupos = tabela[grupo].drop_duplicates().tolist()
    fig, ax = plt.subplots(figsize=(15, max(5, len(grupos) * 0.62 + 2)))
    cores = {'Atrasado': '#C95050', 'No prazo': '#457B9D', 'Sem informação': '#888888'}
    situacoes = [s for s in cores if s in tabela['situacao_entrega'].unique()]
    largura = 0.75 / max(1, len(situacoes))
    for i, situacao in enumerate(situacoes):
        dados = tabela.loc[tabela['situacao_entrega'].eq(situacao)].set_index(grupo).reindex(grupos)
        taxas = dados[metrica].astype(float)
        barras = ax.barh(
            [j + (i - (len(situacoes) - 1) / 2) * largura for j in range(len(grupos))],
            taxas, height=largura, color=cores[situacao], label=situacao
        )
        rotulos = [
            f'{t:.2%}'.replace('.', ',') + f' (n={int(n):,})'.replace(',', '.')
            if pd.notna(t) and pd.notna(n) else ''
            for t, n in zip(taxas, dados[denominador])
        ]
        ax.bar_label(barras, labels=rotulos, padding=3, fontsize=8)
    ax.set_yticks(range(len(grupos)), [str(g).replace('_', ' ') for g in grupos])
    ax.invert_yaxis()
    ax.set_xlim(0, 1.28)
    ax.set_xticks([0, .2, .4, .6, .8, 1])
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.set_xlabel('Proporção')
    ax.set_title(titulo, loc='left', pad=35)
    ax.legend(loc='lower left', bbox_to_anchor=(0, 1), ncol=3, frameon=False)
    ax.grid(axis='x', alpha=.2)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    concluir_grafico(fig, nome, nota)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from pathlib import Path

# Reutiliza a raiz identificada na célula de leitura dos dados.
PASTA_GRAFICOS = RAIZ_PROJETO / 'outputs' / 'graficos'

plt.rcParams['text.parse_math'] = False

def salvar_grafico(fig, arquivo, nota):
    # Reserva espaço para o recorte e salva a mesma figura exibida no notebook.
    fig.text(0.01, 0.01, nota, fontsize=9, color='#555555')
    fig.tight_layout(rect=[0, 0.09, 1, 1])
    registrar_grafico(fig, PASTA_GRAFICOS / arquivo, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)

def barras_taxas(dados, grupo, taxa, quantidade, titulo, arquivo, nota, cores='#C95050'):
    # A ordem das linhas recebidas determina a ordem das barras.
    fig, ax = plt.subplots(figsize=(12, max(5, len(dados) * 0.32 + 2)))
    barras = ax.barh(dados[grupo].astype(str), dados[taxa].astype(float), color=cores)
    ax.invert_yaxis()
    rotulos = [
        f'{t:.2%}'.replace('.', ',') + f' (n = {int(n):,})'.replace(',', '.')
        for t, n in zip(dados[taxa], dados[quantidade])
    ]
    ax.bar_label(barras, labels=rotulos, padding=4, fontsize=9)
    ax.set_title(titulo, loc='left', pad=16)
    ax.set_xlabel('Proporção')
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))
    # Começa em zero; margem à direita para não cortar os rótulos.
    limite = min(1, max(0.05, float(dados[taxa].max()) * 1.4))
    ax.set_xlim(0, limite * 1.18)
    ax.set_xticks([limite * i / 4 for i in range(5)])
    ax.grid(axis='x', alpha=0.2)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    salvar_grafico(fig, arquivo, nota)


In [ ]:
dados = df_comparacao_taxas_recompra.set_index('grupo').loc[['Ruim', 'Neutra', 'Boa']]
fig, ax = plt.subplots(figsize=(10, 6))
barras = ax.bar(dados.index, dados['taxa_recompra'], color=['#C95050', '#E9C46A', '#2A9D8F'], width=0.6)
rotulos = [f'{t:.2%}'.replace('.', ',') + '\n' + f'{int(k):,}/{int(n):,} clientes'.replace(',', '.')
           for t, k, n in zip(dados['taxa_recompra'], dados['clientes_que_recompraram'], dados['total_clientes'])]
ax.bar_label(barras, labels=rotulos, padding=5, fontsize=10,
             bbox={'facecolor': 'white', 'edgecolor': 'none', 'pad': 1}, zorder=5)
referencia = float(resumo_avaliados['taxa_recompra'].iloc[0])
ax.axhline(referencia, color='#555555', linestyle='--', linewidth=1,
           label='Todos os avaliados: ' + f'{referencia:.2%}'.replace('.', ','))
ax.set_title('Recompra observada por avaliação da primeira compra entregue', loc='left', pad=16)
ax.set_xlabel('Avaliação da compra de referência')
ax.set_ylabel('Proporção de clientes que recompraram')
ax.set_ylim(0, max(0.045, float(dados['taxa_recompra'].max()) * 1.6))
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend(frameon=False, loc='upper left')
ax.grid(axis='y', alpha=0.2)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
salvar_grafico(fig, '06_recompra_por_avaliacao.png',
    'Acompanhamento mínimo de 90 dias após a entrega; recompra até o fim da base, com qualquer status.\n'
    'Rótulos: clientes que recompraram / clientes do grupo. Tempos de acompanhamento diferentes; sem interpretação causal.')


In [ ]:
dados = df_comparacao_taxas_recompra.set_index('grupo').loc[['Geral', 'Qualquer avaliação', 'Sem avaliação']]
fig, ax = plt.subplots(figsize=(12, 5))
barras = ax.barh(dados.index, dados['taxa_recompra'], color=['#457B9D', '#2A9D8F', '#888888'])
ax.invert_yaxis()
rotulos = [f'{t:.2%}'.replace('.', ',') + f' ({int(k):,}/{int(n):,} clientes)'.replace(',', '.')
           for t, k, n in zip(dados['taxa_recompra'], dados['clientes_que_recompraram'], dados['total_clientes'])]
ax.bar_label(barras, labels=rotulos, padding=4, fontsize=10)
ax.set_title('Recompra observada: visão geral e presença de avaliação', loc='left', pad=16)
ax.set_xlabel('Proporção de clientes que recompraram')
ax.set_xlim(0, 1.4)
ax.set_xticks([0, 0.25, 0.5, 0.75, 1])
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.grid(axis='x', alpha=0.2)
ax.set_axisbelow(True)
ax.spines[['top', 'right']].set_visible(False)
salvar_grafico(fig, '06_recompra_visao_geral.png',
    'Rótulos: clientes que recompraram / clientes do grupo. Geral inclui clientes com e sem avaliação.\n'
    'Mínimo de 90 dias observáveis após a entrega; recompra sem prazo máximo até o fim da base. Grupos de tamanhos distintos.')


## Exportações

Salva as tabelas em Parquet e CSV e os gráficos em PNG.

In [ ]:
PASTA_PARQUET = RAIZ_PROJETO / 'outputs' / 'tabelas' / 'parquet'
PASTA_CSV = RAIZ_PROJETO / 'outputs' / 'tabelas' / 'csv'
for pasta in (PASTA_PARQUET, PASTA_CSV, PASTA_GRAFICOS):
    pasta.mkdir(parents=True, exist_ok=True)

tabelas_exportar = {
    '06_comparacao_taxas_recompra': df_comparacao_taxas_recompra,
    '06_resumo_tempo_recompra': df_resumo_tempo_recompra,
}
for nome, original in tabelas_exportar.items():
    tabela = original.reset_index(drop=True).convert_dtypes()
    caminho = PASTA_PARQUET / f'{nome}.parquet'
    tabela.to_parquet(caminho, index=False)
    pd.testing.assert_frame_equal(tabela, pd.read_parquet(caminho), check_exact=True)
    tabela.to_csv(
        PASTA_CSV / f'{nome}.csv', index=False,
        sep=';', decimal=',', encoding='utf-8-sig'
    )
    print(f'Tabela exportada em Parquet e CSV: {nome} ({len(tabela)} linhas)')

for nome, conteudo in graficos_exportar.items():
    (PASTA_GRAFICOS / nome).write_bytes(conteudo)
    print(f'Gráfico exportado: {nome}')
